# 01 – Data cleaning

**Canada on Fire (COMP 6934, Group 15).** This notebook turns the three raw downloads into small, clean files that every
figure notebook reads. Run it once, top to bottom (about 2–3 minutes, mostly reading the 2 GB perimeter shapefile).

| Input (`data/raw/`) | Output (`data/clean/`) |
|---|---|
| `NFDB_point_txt/NFDB_point_20260811.txt` – one row per reported fire | `fires.parquet` – cleaned fire points, 1959–2025 |
| `NFDB_poly_large_fires/…large_fires.shp` – burn perimeters ≥ 200 ha, 1972–2024 | `perimeters.parquet` – simplified perimeters (GeoParquet) |
| `lpr_000b21a_e/lpr_000b21a_e.shp` – StatCan 2021 province boundaries | `provinces.parquet` – simplified boundaries (GeoParquet) |
| | `annual_totals.csv` – yearly counts and area, for quick checks |
| | `cleaning_log.csv` – how many rows each step removed |

All map data uses **EPSG:3347** (Statistics Canada Lambert), so every layer lines up without reprojecting.

In [1]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import pyogrio
import shapely

RAW = Path("../data/raw")
CLEAN = Path("../data/clean")
CLEAN.mkdir(parents=True, exist_ok=True)

POINTS_FILE = RAW / "NFDB_point_txt" / "NFDB_point_20260811.txt"
STATS_FILE = RAW / "NFDB_point_txt" / "NFDB_point_stats.xlsx"
POLY_FILE = RAW / "NFDB_poly_large_fires" / "NFDB_poly_1972to2024_20250630_large_fires.shp"
PROV_FILE = RAW / "lpr_000b21a_e" / "lpr_000b21a_e.shp"

CRS = "EPSG:3347"
FIRST_YEAR = 1959   # first year every province and territory reports (see step 2)
LARGE_FIRE_HA = 200

log = []


def record(step, before, after):
    log.append({"step": step, "removed": before - after, "remaining": after})
    print(f"{step:<55} removed {before - after:>7,}  remaining {after:>7,}")

## 1. Load the fire points and check them against the official statistics

NRCan publishes yearly totals for this exact file version (`NFDB_point_stats.xlsx`). If our raw counts and area match theirs
for every year, we know the file was read correctly before we change anything.

In [2]:
raw = pd.read_csv(POINTS_FILE, low_memory=False, encoding="utf-8-sig")
print(f"{len(raw):,} rows, {raw.shape[1]} columns")

official = pd.read_excel(STATS_FILE, sheet_name="NFDB_Summary_Stats", header=5)
official = official[pd.to_numeric(official["YEAR"], errors="coerce").notna()].astype({"YEAR": int})
official = official.set_index("YEAR")[["FIRES", "TOTAL_HA"]]

ours = raw.groupby("YEAR").agg(FIRES=("SIZE_HA", "size"), TOTAL_HA=("SIZE_HA", "sum")).loc[official.index]
assert (ours["FIRES"] == official["FIRES"]).all()
assert np.allclose(ours["TOTAL_HA"], official["TOTAL_HA"], atol=1)
print(f"Counts and area match the official table for all {len(official)} years ({official.index.min()}–{official.index.max()}).")

448,618 rows, 24 columns


Counts and area match the official table for all 67 years (1959–2025).


## 2. Remove records that are not usable wildfires

| Step | Why |
|---|---|
| Exact duplicate rows, and repeats of the same fire ID at the same location and size | Entered twice. (Many *IDs* repeat across different fires, e.g. generic Alberta IDs, so we don't deduplicate on ID alone.) |
| Year −999 or before 1959 | −999 means unknown. Before 1959 only BC, Yukon, Newfoundland and the national parks reported, so national totals would look artificially low. |
| Prescribed burns (`CAUSE2 = H-PB`, `PRESCRIBED = PB`, fire type *PB / Prescribed Burn*) | Deliberately set; not wildfires. |
| Re-burns (`CAUSE2 = RE`) | A fire re-igniting inside an existing burn; counting it again would double-count area. |
| *Mutual Aid / Request for Assistance* (AB, SK) | Responses to fires owned by another agency, which may already be in that agency's records. |

In [3]:
df = raw.copy()
n = len(df)
df = df.drop_duplicates()
df = df.drop_duplicates(subset=["NFDBFIREID", "YEAR", "MONTH", "DAY", "LATITUDE", "LONGITUDE", "SIZE_HA"])
record("Duplicate records", n, len(df))

n = len(df)
df = df[df["YEAR"] >= FIRST_YEAR]
record(f"Unknown year (-999) or before {FIRST_YEAR}", n, len(df))

fire_type = df["FIRE_TYPE"].astype(str).str.strip()
prescribed = (
    (df["CAUSE2"] == "H-PB")
    | (df["PRESCRIBED"].astype(str).str.strip() == "PB")
    | fire_type.isin(["PB", "Prescribed Burn"])
)
n = len(df)
df = df[~prescribed]
record("Prescribed burns", n, len(df))

n = len(df)
df = df[df["CAUSE2"] != "RE"]
record("Re-burns", n, len(df))

assist = df["FIRE_TYPE"].astype(str).str.strip().isin(["Mutual Aid", "Req For Assist", "Request For Ass*"])
n = len(df)
df = df[~assist]
record("Mutual aid / requests for assistance", n, len(df))

Duplicate records                                       removed       9  remaining 448,609
Unknown year (-999) or before 1959                      removed  15,172  remaining 433,437


Prescribed burns                                        removed     437  remaining 433,000
Re-burns                                                removed      74  remaining 432,926


Mutual aid / requests for assistance                    removed     331  remaining 432,595


## 3. Build the analysis columns

- **cause** – `H` → Human, `N` (natural) → Lightning, `U` → Unknown. NRCan codes lightning as `N`, not `L`.
- **date / doy** – built from year, month and day. About 1% of records have no month or day; they get no date but stay in
  yearly totals.
- **decade** – `1960s` … `2020s`. 1959 is the only year in `1950s`, and `2020s` only runs to 2025, so decade comparisons
  should use 1960s–2010s or note the partial decades.
- **size_class** – the Canadian fire size classes: A < 0.1 ha, B 0.1–4, C 4–40, D 40–200, E ≥ 200 ha.
  About 40,000 fires are recorded as 0 ha (tiny spot fires); they count as class A.
- **is_large** – size ≥ 200 ha. Small fires were badly under-reported in early decades, so **use `is_large` rows for any
  trend over time** (as Hanes et al., 2019 do). Large fires are over 97% of all area burned.
- **has_size** – size > 0, for log-scale plots (log of 0 is undefined).

In [4]:
df["cause"] = pd.Categorical(
    df["CAUSE"].map({"H": "Human", "N": "Lightning", "U": "Unknown"}),
    categories=["Human", "Lightning", "Unknown"],
)

month = df["MONTH"].where(df["MONTH"].between(1, 12))
day = df["DAY"].where(df["DAY"].between(1, 31))
df["date"] = pd.to_datetime(dict(year=df["YEAR"], month=month, day=day), errors="coerce")
df["month"] = month.astype("Int64")
df["doy"] = df["date"].dt.dayofyear.astype("Int64")

df["decade"] = (df["YEAR"] // 10 * 10).astype(str) + "s"

df["size_class"] = pd.cut(
    df["SIZE_HA"],
    bins=[-np.inf, 0.1, 4, 40, LARGE_FIRE_HA, np.inf],
    labels=["A", "B", "C", "D", "E"],
    right=False,
)
df["is_large"] = df["SIZE_HA"] >= LARGE_FIRE_HA
df["has_size"] = df["SIZE_HA"] > 0

print(df["cause"].value_counts().to_string())
print(f"\nNo usable month: {df['month'].isna().sum():,}   no full date: {df['date'].isna().sum():,}")
print(f"Large fires: {df['is_large'].sum():,} ({df['is_large'].mean():.1%} of fires, "
      f"{df.loc[df['is_large'], 'SIZE_HA'].sum() / df['SIZE_HA'].sum():.1%} of area)")

cause
Human        231160
Lightning    190632
Unknown       10803

No usable month: 3,554   no full date: 3,878
Large fires: 20,817 (4.8% of fires, 98.9% of area)


## 4. Province boundaries

The StatCan file is very detailed (266 MB, 16.6 million vertices). We simplify it to a 1 km tolerance, which is invisible at
national scale. `preserve_topology=False` is much faster on this many vertices; `make_valid` repairs any shape the
simplification breaks, and the smallest islands disappear, which doesn't matter at this scale. We also attach short province codes and land area for normalising (e.g. area burned per 1,000 km²).

In [5]:
PROVINCE_CODES = {10: "NL", 11: "PE", 12: "NS", 13: "NB", 24: "QC", 35: "ON", 46: "MB",
                  47: "SK", 48: "AB", 59: "BC", 60: "YT", 61: "NT", 62: "NU"}

provinces = gpd.read_file(PROV_FILE).to_crs(CRS)
provinces["province"] = provinces["PRUID"].astype(int).map(PROVINCE_CODES)
provinces = provinces.rename(columns={"PRENAME": "name", "LANDAREA": "land_km2"})
provinces = provinces[["province", "name", "land_km2", "geometry"]]
provinces["geometry"] = shapely.make_valid(provinces.simplify(1000, preserve_topology=False).values)
provinces.to_parquet(CLEAN / "provinces.parquet")
provinces.drop(columns="geometry")

,province,name,land_km2
0,NL,Newfoundland and Labrador,3.581704e+05
1,PE,Prince Edward Island,5.681179e+03
2,NS,Nova Scotia,5.282471e+04
3,NB,New Brunswick,7.124850e+04
4,QC,Quebec,1.298600e+06
5,ON,Ontario,8.924118e+05
6,MB,Manitoba,5.403102e+05
7,SK,Saskatchewan,5.770604e+05
8,AB,Alberta,6.346583e+05
9,BC,British Columbia,9.206866e+05


## 5. Locations: map coordinates and province

243 records have impossible coordinates (e.g. latitude −115); they keep `NaN` map coordinates and are left off maps but
stay in totals. We assign each fire to the province it is **located in**, using its coordinates, rather than the reporting
agency. This places national-park fires (agency `PC`) in their province. Fires just off the simplified coastline are matched
to the nearest province within 10 km. The rare fire still unmatched falls back to its reporting agency.

In [6]:
valid_xy = df["LATITUDE"].between(41, 84) & df["LONGITUDE"].between(-142, -52)
print(f"Invalid coordinates: {(~valid_xy).sum():,}")

points = gpd.GeoDataFrame(
    df.loc[valid_xy, []],
    geometry=gpd.points_from_xy(df.loc[valid_xy, "LONGITUDE"], df.loc[valid_xy, "LATITUDE"]),
    crs="EPSG:4326",
).to_crs(CRS)

inside = gpd.sjoin(points, provinces[["province", "geometry"]], how="left", predicate="within")
inside = inside[~inside.index.duplicated()]
missing = inside["province"].isna()
nearest = gpd.sjoin_nearest(points[missing], provinces[["province", "geometry"]], how="left", max_distance=10_000)
nearest = nearest[~nearest.index.duplicated()]
inside.loc[missing, "province"] = nearest["province"]

df["x"] = points.geometry.x
df["y"] = points.geometry.y
df["province"] = inside["province"]
fallback = df["province"].isna() & df["SRC_AGENCY"].ne("PC")
df.loc[fallback, "province"] = df.loc[fallback, "SRC_AGENCY"]
print(f"Placed by location: {valid_xy.sum() - missing.sum():,}   nearest within 10 km: "
      f"{nearest['province'].notna().sum():,}   by agency: {fallback.sum():,}   "
      f"unplaced: {df['province'].isna().sum():,}")

Invalid coordinates: 207


Placed by location: 426,912   nearest within 10 km: 5,354   by agency: 195   unplaced: 134


## 6. Save the clean fire table

In [7]:
fires = df.rename(columns={
    "NFDBFIREID": "fire_id", "SRC_AGENCY": "agency", "YEAR": "year", "LATITUDE": "lat",
    "LONGITUDE": "lon", "SIZE_HA": "size_ha",
})
fires.loc[~valid_xy, ["lat", "lon"]] = np.nan
fires = fires[["fire_id", "agency", "province", "year", "month", "date", "doy", "decade", "cause",
               "size_ha", "size_class", "is_large", "has_size", "lat", "lon", "x", "y"]]
fires = fires.reset_index(drop=True)
fires.to_parquet(CLEAN / "fires.parquet", index=False)
print(f"Saved {len(fires):,} fires, {fires['year'].min()}–{fires['year'].max()}")
fires.head()

Saved 432,595 fires, 1959–2025


,fire_id,agency,province,year,month,date,doy,decade,cause,size_ha,size_class,is_large,has_size,lat,lon,x,y
0,AB-2024-CWF-001-2024,AB,AB,2024,1,2024-01-02,2,2020s,Human,0.05,A,False,True,50.066333,-114.154883,4.642932e+06,1.822978e+06
1,AB-2024-HWF-001-2024,AB,AB,2024,1,2024-01-05,5,2020s,Human,0.40,B,False,True,57.912833,-116.334050,4.820492e+06,2.675607e+06
2,AB-2024-SWF-001-2024,AB,AB,2024,1,2024-01-17,17,2020s,Human,0.01,A,False,True,56.575550,-115.216533,4.828388e+06,2.516196e+06
3,AB-2024-LWF-001-2024,AB,AB,2024,1,2024-01-09,9,2020s,Human,0.01,A,False,True,55.957361,-110.709667,5.064703e+06,2.363642e+06
4,AB-2024-LWF-002-2024,AB,AB,2024,1,2024-01-09,9,2020s,Human,0.01,A,False,True,55.957361,-110.709667,5.064703e+06,2.363642e+06


## 7. Burn perimeters (fires ≥ 200 ha)

The shapefile is 2.1 GB because each polygon traces the burn edge in fine detail and carries an unused Z (height) value.
We drop the Z value, remove records with no year (−9999) and prescribed burns, and simplify to a 250 m tolerance. That
keeps the shape of every fire visible on a national or provincial map while making the file small enough to plot quickly.

In [8]:
perims = pyogrio.read_dataframe(POLY_FILE, use_arrow=True, columns=[
    "CFS_REF_ID", "SRC_AGENCY", "YEAR", "MONTH", "DAY", "SIZE_HA", "CALC_HA", "CAUSE"])
n = len(perims)
perims = perims[(perims["YEAR"] > 0) & (perims["CAUSE"] != "H-PB")]
print(f"Perimeters: {n:,} -> {len(perims):,} after removing unknown years and prescribed burns")

perims["geometry"] = shapely.force_2d(perims.geometry.values)
perims = perims.to_crs(CRS)
perims["geometry"] = shapely.make_valid(perims.simplify(250, preserve_topology=False).values)
perims = perims[~perims.is_empty & perims.geometry.notna()]

perims = perims.rename(columns={"CFS_REF_ID": "fire_id", "SRC_AGENCY": "agency", "YEAR": "year",
                                "MONTH": "month", "DAY": "day", "SIZE_HA": "size_ha", "CALC_HA": "calc_ha"})
perims["cause"] = perims.pop("CAUSE").map({"H": "Human", "N": "Lightning", "U": "Unknown"})
perims["decade"] = (perims["year"] // 10 * 10).astype(str) + "s"
perims.to_parquet(CLEAN / "perimeters.parquet")
print(f"Saved {len(perims):,} perimeters, {perims['year'].min()}–{perims['year'].max()}")

Perimeters: 16,394 -> 16,266 after removing unknown years and prescribed burns


/var/folders/yw/vsrrtf414tsg5q4jkzxdcyn40000gn/T/ipykernel_69703/744728906.py:10: UserWarning: GeoSeries.notna() previously returned False for both missing (None) and empty geometries. Now, it only returns False for missing values. Since the calling GeoSeries contains empty geometries, the result has changed compared to previous versions of GeoPandas.
Given a GeoSeries 's', you can use '~s.is_empty & s.notna()' to get back the old behaviour.

To further ignore this warning, you can do: 
import warnings; warnings.filterwarnings('ignore', 'GeoSeries.notna', UserWarning)
  perims = perims[~perims.is_empty & perims.geometry.notna()]


Saved 16,262 perimeters, 1972–2024


## 8. Summary and checks

`annual_totals.csv` gives yearly fire counts and area, overall and for large fires. It is a quick reference while writing
the report; the figures should still be built from `fires.parquet`.

In [9]:
annual = fires.groupby("year").agg(
    fires=("size_ha", "size"),
    area_ha=("size_ha", "sum"),
    large_fires=("is_large", "sum"),
    large_area_ha=("size_ha", lambda s: s[fires.loc[s.index, "is_large"]].sum()),
)
annual.to_csv(CLEAN / "annual_totals.csv")

log_table = pd.DataFrame(log)
log_table.to_csv(CLEAN / "cleaning_log.csv", index=False)
display(log_table)

top = annual["area_ha"].nlargest(5)
print("Largest years by area burned (Mha):", ", ".join(f"{y}: {a / 1e6:.1f}" for y, a in top.items()))
assert top.index[0] == 2023

for f in sorted(CLEAN.iterdir()):
    print(f"{f.name:<22} {f.stat().st_size / 1e6:6.1f} MB")

,step,removed,remaining
0,Duplicate records,9,448609
1,Unknown year (-999) or before 1959,15172,433437
2,Prescribed burns,437,433000
3,Re-burns,74,432926
4,Mutual aid / requests for assistance,331,432595


Largest years by area burned (Mha): 2023: 17.6, 2025: 8.7, 1989: 7.6, 1995: 7.5, 1981: 6.3
annual_totals.csv         0.0 MB
cleaning_log.csv          0.0 MB
fires.parquet            16.6 MB
perimeters.parquet       23.0 MB
provinces.parquet         0.7 MB
